# Payment Gateway Performance & Risk Analytics

**Key finding:** Chargebacks, not gateway or payment-method failures, are the biggest profit leak (loss of Rs 464.6K vs Rs 241.8K fee revenue). The full Executive Summary is at the end of this notebook.

*Data: synthetic payments dataset, 50,000 raw rows, Jan-Jun 2026.*

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

df = pd.read_csv("payment_cleaned.csv")
df.head()

,transaction_id,timestamp,user_id,device_id,merchant_name,merchant_category,gateway_name,amount,currency,payment_method,...,response_time_ms,settlement_id,settled_amount,mdr_fee,settlement_status,settled_date,risk_score,chargeback_flag,refund_status,refund_amount
0,TXN_0021367,2026-05-17 10:09:37,USR_26248,DEV_25264,Meesho,E-commerce,PayU,1007.22,INR,UPI,...,336.0,NaN,NaN,20.14,Not Applicable,NaN,31.54,0,Not Refunded,0.0
1,TXN_0003228,2026-02-07 12:51:27,USR_16739,DEV_21488,Flipkart,E-commerce,Razorpay,130.32,INR,UPI,...,1411.0,SET_916887,130.32,1.95,Settled,2026-02-11 12:51:27,30.00,0,Not Refunded,0.0
2,TXN_0047958,2026-04-03 09:35:22,USR_19282,DEV_24379,BookMyShow,Entertainment,PayU,102.26,INR,UPI,...,424.0,SET_740417,102.26,1.23,Settled,2026-04-05 09:35:22,47.16,0,Not Refunded,0.0
3,TXN_0010902,2026-02-09 02:05:36,USR_22881,DEV_29644,Nykaa,Beauty,Razorpay,705.13,INR,CREDIT CARD,...,151.0,SET_200209,705.13,8.46,Settled,2026-02-12 02:05:36,37.34,0,Not Refunded,0.0
4,TXN_0032753,2026-03-01 03:00:54,USR_25648,DEV_15551,Amazon,E-commerce,Cashfree,262.43,INR,UPI,...,323.0,SET_830870,262.43,4.72,Pending,NaN,84.54,0,Not Refunded,0.0


In [3]:
# Q1: What is total transactions volumme and value
total_transactions = len(df)
total_amount = df['amount'].sum()

print(f"Total transactions:{total_transactions}")
print(f"Total Amount processed:INR{total_amount:,.2f}")

Total transactions:49820
Total Amount processed:INR20,205,710.70


# insight: the platform processed 49820 transaction worth 2.02cr 

In [5]:
# Q2: what is the overall success rate
status_counts = df['status'].value_counts()
success_rate = (status_counts['Success'] / len(df)) * 100

print(f"Successful Transaction: {status_counts['Success']}")
print(f"Success Rate: {success_rate:.2f}%")

Successful Transaction: 38893
Success Rate: 78.07%


## insight: 78% of all transaction were successful(38893 out of 49,820) remaining 22% did not successed (failed, pending, or refunded)
# action: break down the failure reason in detail

In [7]:
# Q3: what is the average transaction amount( mean vs median)
avg_amount = df['amount'].mean()
median_amount = df['amount'].median()

print(f"Average amount (Mean): {avg_amount:,.2f}")
print(f"Median amount: {median_amount:,.2f}")

Average amount (Mean): 405.57
Median amount: 243.44


# insight: the mean amount(405.57) is higher than the median amount(243.44).a large amount pull the mean and most are smaller amounnt.
# business impact: using mean alone in reports could overstate typical customer spending leading to inaccurate assumptions in priceing market target and revenue
# action: use the median rather than the mean, as it better represents actual customer behaviour

In [9]:
## Q4: Payment method performance(transaction, total value, average value)
payment_summary = (
    df.groupby('payment_method')
    .agg(
        transactions=('transaction_id', 'count'),
        total_amount=('amount', 'sum'),
        avg_amount=('amount', 'mean')
    )
    .sort_values('transactions', ascending=False)
)

payment_summary['share_%'] = (payment_summary['transactions'] / len(df) * 100)
payment_summary.round(2)

,transactions,total_amount,avg_amount,share_%
payment_method,,,,
UPI,23899,9677674.70,404.94,47.97
CREDIT CARD,9909,4019253.57,405.62,19.89
DEBIT CARD,8090,3310787.46,409.24,16.24
NET BANKING,4952,1974043.76,398.64,9.94
WALLET,2970,1223951.21,412.10,5.96


# insight: upi leads in transaction(48%) but average transaction value in nearly identical across all payment methods and not higher value purchase
# impact: the dependency on upi is a revenue concentration risk, not just a traffic one any upi failure directly half of processed value
# action: track upi success rate and uptimme as a priority kpi and compare success rate by payment method in the next these numbers include fail and pending trans.

In [11]:
# Q5: transactions, total amount and average amount by merchant category

merchant_summary = (
    df.groupby('merchant_category')
    .agg(
        transactions=('transaction_id', 'count'),
        total_amount=('amount', 'sum'),
        avg_amount=('amount', 'mean')
    )
    .sort_values('transactions', ascending=False)
)

merchant_summary['share_%'] = (merchant_summary['transactions'] / len(df) * 100)
merchant_summary.round(2)


,transactions,total_amount,avg_amount,share_%
merchant_category,,,,
E-commerce,19727,8016965.04,406.40,39.60
Food Delivery,11029,4414042.59,400.22,22.14
Fashion,4979,2008345.39,403.36,9.99
Beauty,3970,1594748.06,401.70,7.97
Entertainment,3601,1500557.94,416.71,7.23
Travel,3492,1426009.38,408.36,7.01
Financial Services,3022,1245042.30,411.99,6.07


# insight: E-commerce(40%) and food delivery(22%) total 62% of all transaction while average transaction value remain consistent across all category
# busniess impact: heavy reliance on just two merchant categories create concentration risk any disruption eccomrce and food delivery would impact overall busnies volume
# action: 
# track the success rate of eccomerce and food delivery as seprate priority KPIs
# analyse success rate by merchant category to check any large category has a higher failure rate
# treat smaller categories as growth opportunities to diversify revenue.

# amount include failed and pending transaction, so they represent attempted value, not realised revenue


In [13]:
# Q6: which payment gateway has the highest success rate.

# Flag successful transaction(1=success, 0=otherwise)
df['is_success'] = (df['status'] == 'Success').astype(int)

# Gateway level summary
gateway_summary = (
    df.groupby('gateway_name')
    .agg(
        transactions=('transaction_id', 'count'),
        successful_txns=('is_success', 'sum'),
        success_rate=('is_success', 'mean')
    )
)

# convert success rate to percentage
gateway_summary['success_rate'] = gateway_summary['success_rate'] * 100

# Gap versus overall success rate(percentage points)
overall_rate = df['is_success'].mean() * 100
gateway_summary['vs_overall_pp'] = gateway_summary['success_rate'] -overall_rate

# best to worst
gateway_summary.sort_values('success_rate', ascending=False).round(2)
    

,transactions,successful_txns,success_rate,vs_overall_pp
gateway_name,,,,
Unknown,498,393,78.92,0.85
PayU,11353,8890,78.31,0.24
Cashfree,13274,10385,78.24,0.17
Razorpay,19712,15388,78.06,-0.00
Stripe,4983,3837,77.00,-1.07


# insight: all gateway show similar success rate
#. stripe is the lowest at 1.07pp below overall rate.but sample is small(4983 transaction) so the gaplis not meaning full
#. this confirm that failure are not gateway specific.


# impact: razorpay high transaction volume does'nt  come with performance tradeoff its success rate is near overall average
# so the main risk from razorpay dependency is concentration, not reliability

# action: investigate the root cause beyond the gateway level (failure reason, payment, issueer bank)
# track gateway success rate monthly and keep monitoring

# note = unknown(498 transaction, missing gateway) is excluded from comparison

In [15]:
# Q6 B: Realized revenue vs attempted value

# total value of all attempted transactions
total_attempted = df['amount'].sum()

# value of successful transaction only
realized_revenue = df.loc[df['status'] == 'Success', 'amount'].sum()

# revenue gap and realized share
revenue_gap = total_attempted-realized_revenue
realized_pct = realized_revenue / total_attempted * 100

print(f"Total attempted value: INR {total_attempted:,.2f}")
print(f"Realized revenue: INR {realized_revenue:,.2f}")
print(f"Revenue Gap: INR {revenue_gap:,.2f}")
print(f"Realized share: {realized_pct:.2f}%")





Total attempted value: INR 20,205,710.70
Realized revenue: INR 15,727,325.38
Revenue Gap: INR 4,478,385.32
Realized share: 77.84%


# insight: out of INR2.02 crore in attempted transaction value, only INR1.57 crore(77.84%) was actually revenue a gap of iNR 44.78lakh lost to fail and pending refund transaction
# business impact: this gap represent real, quantifiable revenue loss. even small improvements in success rate translate directly into meaningful revenue gain.
# action: set a target to reduce the revenue gap(currently 22.16%) by addressing top failure reasons - bank decline and insufficient fund for large share of fail attempt

In [17]:
# Q7: (a) Failure reason breakdown

df.groupby('status')['failure_reason'].count()


status
Failed      6855
Pending      893
Refunded       0
Success        0
Name: failure_reason, dtype: int64

In [18]:
# validation-failure_reason populated by status.


In [19]:
# Q7(b) : failure reason breakdown for failed transactions

# keep only failed transactions
failed_df = df[df['status'] == 'Failed']

# summary by failure reason (dropna=False keeps missing reasons visible)

failure_summary = (
    failed_df.groupby('failure_reason', dropna=False)
    .agg(
        failed_txns=('transaction_id', 'count'),
        failed_amount=('amount', 'sum')
    )
    .sort_values('failed_txns', ascending=False)
)

# share of each reason within failed transactions

failure_summary['share_%'] = failure_summary['failed_txns'] / len(failed_df) * 100

failure_summary.round(2)
                                                                  

                                                                  





                                                                  
                                                                  

,failed_txns,failed_amount,share_%
failure_reason,,,
Bank Declined,1704,687538.17,24.86
Insufficient Funds,1653,705043.61,24.11
Timeout,1119,450927.65,16.32
Technical Error,833,332501.02,12.15
Invalid OTP,675,278622.99,9.85
Limit Exceeded,557,216354.34,8.13
Network Error,314,129635.57,4.58


# insight: 
# .bank declined(24.86%) and insufficient Funds(24.11)% are two largest failure reasons, together accounting(49%) of failed transactionn
# .customer/bank side reason(bank declined, insufficient funds, invalid otp, limit exceeded) cause 66.94% of failure(inr 18.9 lakh), while system side
# reason(time out, technical error, network error) cause 33.06% (inr 9.1lakh)
# timeout is the largest system side issue (16.32%) of all failres

# business impact: 
# one in three failed transaction is caused by technical or network problem that the busniess can directly influence.
# eliminating all system side failures could lift the overall success rate up to 4.5pp from to 78.07% to 82.6%, recover to 9 lakh attempted value
# even then, the rate stays well below the 95% benchwork so customer/bamk side needs attention.

# action:
# priortise fixing time out and technical error with the gateway/engineering teams(system side , controllable)
# for insufficient funds and invalid otp, improve the customer exprience: clear error messaage, easy retry, fall back another payment method
# monitoring bamk decline by issuer bank to check whether a few bamks drive most of the declines


In [21]:
# 7(c): which failure reasons appear on pending
df[df['status'] == 'Pending']['failure_reason'].value_counts()

failure_reason
Timeout          495
Network Error    398
Name: count, dtype: int64

# 893 pending transaction carry a failure reason and all of them are system side.this indicates pending status is driven by connectivity issue rather than
# customer or bamk decision. pending transaction may still settle later, so their outcome should be verified in the settlement analysis


In [23]:
#Q7(d): group failure reason into system side vs customer/bank side

# step1: only failed transaction (.copy() avoids a warning when adding a column)
failed_df = df[df['status'] =='Failed'].copy()

# step2: list of systeem side reasons
system_side = ['Timeout', 'Technical Error', 'Network Error']

# step 3: New column label each row as system side or customer/bank side
failed_df['failure_side'] = np.where(
    failed_df['failure_reason'].isin(system_side),
    'System side',
    'Customer/Bank side'
)

# step 4: summary by side
side_summary = (
    failed_df.groupby('failure_side')
    .agg(
        failed_txns=('transaction_id', 'count'),
        failed_amount=('amount', 'sum')
    )
    .sort_values('failed_txns', ascending=False)
)

# step 5 share within failed transactions
side_summary['share%'] = side_summary['failed_txns']/len(failed_df) * 100

side_summary.round(2)

,failed_txns,failed_amount,share%
failure_side,,,
Customer/Bank side,4589,1887559.11,66.94
System side,2266,913064.24,33.06


### Q7 (d): Failure Side Analysis - Controllable vs Non-Controllable



# Insight
# Customer/bank-side reasons (Bank Declined, Insufficient Funds, Invalid OTP, Limit Exceeded) account for 66.94% of failed transactions (4,589 transactions, INR 18.9 lakh).
# System-side reasons (Timeout, Technical Error, Network Error) account for 33.06% (2,266 transactions, INR 9.1 lakh).
# All 893 Pending transactions that carry a failure reason are also system-side (Timeout: 495, Network Error: 398).

# Business Impact
# One in three failed transactions is caused by technical or network issues that the business can directly influence.
# Including Pending, system-side issues affected at least 3,159 transactions (about 6.3% of all transactions).
# Eliminating all system-side failures could lift the overall success rate by up to ~4.5 pp (78.07% to ~82.6%), recovering up to ~INR 9.1 lakh of attempted value. This is an upper bound, since not every failed payment would be retried.
# Even then, the rate stays below the 95% benchmark, so customer/bank-side failures also need attention.

# Recommended Action
# Prioritise fixing Timeout and Technical Error with the gateway and engineering teams.
# For Insufficient Funds and Invalid OTP, improve the customer experience with clear error messages, easy retry and a fallback payment method.
# Monitor Bank Declined by issuer bank to check whether a few banks drive most declines.
# Track the system-side failure share monthly as a KPI.
# Verify in the settlement analysis whether Pending transactions eventually settle.

In [25]:
# Q8: response time by gateway

response_summary =(
    df.groupby('gateway_name')
    .agg(
        transactions=('transaction_id', 'count'),
        avg_ms=('response_time_ms', 'mean'),
        median_ms=('response_time_ms', 'median')
)
.sort_values('avg_ms')
)

response_summary.round(1)

,transactions,avg_ms,median_ms
gateway_name,,,
Unknown,498,703.0,636.5
Razorpay,19712,716.0,661.0
Cashfree,13274,717.3,661.0
PayU,11353,717.5,661.0
Stripe,4983,725.5,661.0


# insight: gateway speed is virtually identical: average response time ranges only from 716 ms(Razorpay) to 725.5 ms(stripe) a gap to 9.5ms and median 661 ms for all
# the typical payment takes about 0.66 seconds.the average 55ms above the median because small number of slow transaction 

# impact: response time does not differentiate gateways, so switching or re-routing traffic between gateways would not improve payment speed.
# Combined with question 6 (similar success rates) and question 7 (timeout being the largest system-side failure), this suggests the slowness and failures are platform-wide rather than caused by one gateway.


# action: do not re-route traffic between gateways for Speed reasons.
# Investigate what drives slow transactions (mere  payment method, issuer bank, time of day) and whether slow responses are linked to timeout failures. 
# Track median and tail (P95) response time per gateway monthly. 

# Note: "Unknown", (498 transactions, missing gateway is excluded from the vendor comparison). 



### Q9: Settlement analysis

**objective**: understand how transaction move into settlement and measure how long settlement takes, to identify cash-flow delays for merchants.







In [28]:
# Q9(a): inspect settlement-related columns

df[['timestamp', 'settled_date', 'settlement_status']].head(10)


,timestamp,settled_date,settlement_status
0,2026-05-17 10:09:37,NaN,Not Applicable
1,2026-02-07 12:51:27,2026-02-11 12:51:27,Settled
2,2026-04-03 09:35:22,2026-04-05 09:35:22,Settled
3,2026-02-09 02:05:36,2026-02-12 02:05:36,Settled
4,2026-03-01 03:00:54,NaN,Pending
5,2026-05-09 00:51:31,2026-05-11 00:51:31,Settled
6,2026-04-24 09:52:54,2026-04-26 09:52:54,Settled
7,2026-03-10 10:07:05,2026-03-14 10:07:05,Settled
8,2026-04-29 09:33:08,2026-05-03 09:33:08,Settled
9,2026-04-26 05:51:21,2026-04-27 05:51:21,Settled


In [29]:
#  Q9(b): Distribution of settlement status

df['settlement_status'].value_counts()

settlement_status
Settled           36576
Not Applicable     6855
Pending            5609
Failed              780
Name: count, dtype: int64

In [30]:
# Q9(c): Relationship between transaction status and settlement status

pd.crosstab(df['status'], df['settlement_status'])

settlement_status,Failed,Not Applicable,Pending,Settled
status,,,,
Failed,0,6855,0,0
Pending,0,0,2546,0
Refunded,0,0,299,1227
Success,780,0,2764,35349


**Settlement Overview**
- All 6,855 Failed transactions have settlement status "Not Applicable", as expected, since a failed payment has nothing to settle.
- All 2,546 Pending transactions are also Pending in settlement; none of them has settled in this dataset.
- Of 38,893 Successful transactions, 35,349 (90.9%) are Settled, 2,764 (7.1%) are still Pending settlement and 780 (2.0%) have a failed settlement.
- "Failed" in `settlement_status` (780) is different from "Failed" in `status` (6,855): the payment succeeded, but the payout to the merchant failed.

**Data note:** 36,576 rows are marked "Settled", but `settled_date` is populated for only 36,209 rows. This gap is validated below before calculating settlement delay.

In [32]:
# Q9(d): settled rows with missing settled_date

((df['settlement_status'] == 'Settled') & (df['settled_date'].isna())).sum()

367

** Data note:**367 rows are marked 'Settled' but have no 'settle_date'. These rows are excluded from the settlement delay calculation, leaving 36, 209 rows.
the gap should be raised with the data or settlement team as possible pipeline issue.

In [34]:
# Q9(e): convert to datetime and calculate settlement delays in days

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['settled_date'] = pd.to_datetime(df['settled_date'])
df['settlement_days'] = (df['settled_date'] - df['timestamp']).dt.days

In [35]:
#Q9(f): validation - check settlement_days  for invalid values
df['settlement_days'].describe()

count    36209.000000
mean         2.500815
std          1.118432
min          1.000000
25%          2.000000
50%          2.000000
75%          4.000000
max          4.000000
Name: settlement_days, dtype: float64

**validation note:**Settlement delay is available for 36,209 transaction and ranges from 1 to 4 days, with no negative or extreme values. the median is 2 days and 
                                                                                   the mean is 2.5 days. the 75 percentile is 4 days, so at least a quarter of settlements
                                                                                   ments take twice as long as the median
                                                                                  

In [37]:
#Q9(g): distribution of settlement delay(days)
df['settlement_days'].value_counts().sort_index()

settlement_days
1.0    9004
2.0    9183
3.0    8906
4.0    9116
Name: count, dtype: int64

**Observation:**Settlement delay is spread almost evenly 1, 2, 3, 4 days(about 25%).There is no standard settlement cycle, so merchants cannot predict when they
will be paid

In [39]:
# Q9(H): settlement delay (days) by payment gateway
settlement_summary = (
     df.groupby('gateway_name')
     .agg(
         settled_txns=('settlement_days', 'count'),
         avg_days=('settlement_days', 'mean'),
         median_days=('settlement_days', 'median'),
         std_days=('settlement_days', 'std')
     )
     .sort_values('avg_days')
 )

settlement_summary.round(2)
                 

,settled_txns,avg_days,median_days,std_days
gateway_name,,,,
Stripe,3604,2.48,2.0,1.11
PayU,8271,2.50,2.0,1.12
Razorpay,14308,2.50,2.0,1.11
Unknown,367,2.52,2.0,1.13
Cashfree,9659,2.52,3.0,1.13


**Insight**
- Settlement time is virtually identical across gateways: the average ranges only from 2.48 days (Stripe) to 2.52 days (Cashfree), a gap of about one hour, and the spread (std of about 1.1 days) is the same for all of them.
- Cashfree's median of 3 days versus 2 days for the others is an artefact of whole-day data around the 2/3-day boundary, not a real difference.

**Business Impact**
- Gateway choice does not explain settlement delays, so the delay appears to be platform-wide rather than gateway-specific.

**Recommended Action**
- Do not re-route traffic between gateways for settlement-speed reasons.
- Investigate settlement processing at the platform level.

*Note: "Unknown" (367 settled transactions, missing gateway) is excluded from the vendor comparison.*

In [41]:
# Q10 (a): Extract hour and day name from timestamp
df['hour'] = df['timestamp'].dt.hour
df['day_name'] = df['timestamp'].dt.day_name()

# Transactions by hour
df['hour'].value_counts().sort_index()

hour
0     2187
1     2009
2     2132
3     2086
4     2123
5     2106
6     1998
7     1979
8     2119
9     2118
10    2150
11    2079
12    2065
13    2017
14    2008
15    2088
16    2046
17    2117
18    2067
19    2100
20    2106
21    2012
22    2045
23    2063
Name: count, dtype: int64

### Q10 (a): Transaction Volume by Hour

**Observation**
- Transaction volume is almost flat across the day, ranging from 1,979 (07:00) to 2,187 (00:00) per hour, against an average of about 2,076.
- The highest and lowest hours deviate by only about ±5% from the average, so no meaningful peak or off-peak window exists.

**Insight**
- Hourly volume alone does not show any load-driven pattern. Real payment platforms typically show evening peaks and night troughs, so this uniform distribution suggests the dataset is synthetic.

**Business Impact**
- Peak-hour capacity planning cannot be justified from this data. Any failure spikes, if they exist, would need to be explained by factors other than traffic load.

**Next Action**
- Check whether the success rate varies by hour (Q10 (b)) and by day of week (Q10 (c)) before concluding that time has no effect.

In [43]:
# Q10 (b): Hourly volume and success rate
hourly_summary = (
    df.groupby('hour')
      .agg(
          transactions=('is_success', 'count'),
          success_rate_pct=('is_success', 'mean')
      )
)
hourly_summary['success_rate_pct'] = (hourly_summary['success_rate_pct'] * 100).round(2)
hourly_summary

,transactions,success_rate_pct
hour,,
0,2187,80.06
1,2009,78.25
2,2132,76.92
3,2086,77.56
4,2123,78.00
5,2106,76.97
6,1998,77.53
7,1979,79.33
8,2119,80.37


**Observation**
- The hourly success rate ranges from 75.78% (09:00) to 80.37% (08:00), against an overall average of about 78.1%.
- The best and worst hours are adjacent (08:00 and 09:00), and no sustained low-performing window appears across consecutive hours.

**Insight**
- With roughly 2,000 transactions per hour, a random variation of about ±1 percentage point is expected, so a spread of ±2.3 pp across 24 hours is consistent with chance rather than a true time-of-day effect.
- Hour of day is therefore unlikely to be a driver of payment failures.

**Business Impact**
- Time-based interventions (hour-specific alerts, scheduled capacity changes or maintenance windows) are not justified by this data.
- Combined with the flat volume pattern, no peak-hour risk is visible.

**Recommended Action**
- Do not prioritise time-of-day optimisation. Shift the investigation to other drivers: payment method and issuer bank (Bank Declined was the top failure reason).
- Before finalising, confirm the day-of-week view (Q10 (c)) shows the same absence of pattern.

In [45]:
# Q10 (c): Day-of-week volume and success rate
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']

daily_summary = (
    df.groupby('day_name')
      .agg(
          transactions=('is_success', 'count'),
          success_rate_pct=('is_success', 'mean')
      )
      .reindex(day_order)
)
daily_summary['success_rate_pct'] = (daily_summary['success_rate_pct'] * 100).round(2)
daily_summary

,transactions,success_rate_pct
day_name,,
Monday,7214,78.62
Tuesday,7224,78.60
Wednesday,6815,77.90
Thursday,7214,78.38
Friday,7218,77.35
Saturday,7117,77.87
Sunday,7018,77.73


In [46]:
# Validation: date range of the dataset
df['timestamp'].min(), df['timestamp'].max()

(Timestamp('2026-01-01 00:00:37'), Timestamp('2026-06-30 23:46:12'))

### Q10 (c): Day-of-Week Pattern

**Observation**
- The success rate ranges from 77.35% (Friday) to 78.62% (Monday), a spread of only 1.27 percentage points.
- Wednesday shows the lowest total volume (6,815), but the data period (1 Jan to 30 Jun 2026) contains 25 Wednesdays versus 26 of every other weekday.

**Insight**
- Normalised for the number of days, volume is stable at about 270-278 transactions per day for every weekday, so the apparent Wednesday dip is a calendar effect, not a change in customer behaviour.
- With about 7,100 transactions per weekday, random variation of around ±0.5 pp is expected, so the day-wise differences in success rate are not meaningful. Day of week is not a driver of failures.

**Business Impact**
- Even if Friday performed like Monday, the gain would be only about 90 transactions over the six months, so a day-specific intervention is not justified.

**Recommended Action**
- Do not prioritise day-of-week optimisation.
- When comparing days, always normalise volume by the number of occurrences of each weekday in the period.

---

### Q10 Summary: Time-Based Analysis
Neither hour of day nor day of week explains payment failures. Volume and success rates are stable, with differences consistent with random variation. The investigation now moves to other drivers: **payment method** and **issuer bank**.

In [48]:
df['payment_method'].value_counts()

payment_method
UPI            23899
CREDIT CARD     9909
DEBIT CARD      8090
NET BANKING     4952
WALLET          2970
Name: count, dtype: int64

In [49]:
### Q11(a): Success rate and volumme share by payment method

summary = (
    df.groupby('payment_method')
    .agg(
        transactions=('is_success', 'count'),
        success_rate=('is_success', 'mean')
    )
    .sort_values('success_rate')

)

summary['success_rate'] = (summary['success_rate']*100).round(2)
summary['share_%']=(summary['transactions'] / len(df) * 100).round(2)
                   
summary

,transactions,success_rate,share_%
payment_method,,,
DEBIT CARD,8090,77.39,16.24
CREDIT CARD,9909,77.72,19.89
WALLET,2970,77.78,5.96
UPI,23899,78.37,47.97
NET BANKING,4952,78.59,9.94


**Insight**
- Success rate is almost the same across all payment methods, from 77.39% (Debit Card) to 78.59% (Net Banking). The overall rate is about 78%.
- The 1.2 pp gap is small enough to be random variation, so payment method is not a driver of failures.

**Business Impact**
- If every method matched the best one, only about 260 extra transactions would succeed, which is very small compared with the potential from fixing system-side failures.
- UPI carries 48% of all traffic, so any UPI-specific problem would affect nearly half of all payments.

**Recommended Action**
- Do not prioritise method-specific fixes.
- Next, check success rate by issuer bank, since Bank Declined is the top failure reason.

In [51]:
# Issuer bank wise success rate

# Q12(a): check issuer_bank categories and missing values

df['issuer_bank'].value_counts(dropna=False)

issuer_bank
HDFC Bank     11420
SBI           10865
ICICI Bank     9803
Axis Bank      8403
Kotak Bank     4846
Yes Bank       3988
Unknown         495
Name: count, dtype: int64

### Q12 (b): Success Rate by Issuer Bank

**Business Question**
- Are payment failures, especially Bank Declined, concentrated in specific issuer banks?

**Why It Matters**
- Bank Declined is the largest failure reason. If one bank performs much worse than the others, it can be raised with that bank or handled with a retry flow. If all banks are similar, the issue is platform-wide.

In [53]:
# Q12(b): success rate and volume share by issuer bank

bank_summary =(
    df.groupby('issuer_bank')
    .agg(
        transactions=('is_success', 'count'),
        success_rate_pct=('is_success', 'mean')
    )
     .sort_values('success_rate_pct')
)

bank_summary['success_rate_pct'] = (bank_summary['success_rate_pct']*100).round(2)
bank_summary['share_%'] = (bank_summary['transactions'] / len(df)*100).round(2)
bank_summary


,transactions,success_rate_pct,share_%
issuer_bank,,,
Unknown,495,74.75,0.99
Yes Bank,3988,77.76,8.00
SBI,10865,77.87,21.81
ICICI Bank,9803,78.00,19.68
HDFC Bank,11420,78.11,22.92
Axis Bank,8403,78.23,16.87
Kotak Bank,4846,78.85,9.73


**Insight**
- Success rates are almost identical across the six named banks, from 77.76% (Yes Bank) to 78.85% (Kotak Bank). The 1.09 pp gap is within random variation, so no bank stands out as a problem.
- "Unknown" (495 transactions, 74.75%) is too small and is not a real bank, so it is excluded from the comparison.

**Business Impact**
- If all banks matched the best one, only about 370 extra transactions (about 0.7 pp) would succeed, so bank-specific escalation is not justified.
- Bank Declined appears to be spread evenly across issuers rather than caused by one bank.

**Recommended Action**
- Do not prioritise bank-specific fixes.
- Investigate the failure reasons themselves, starting with system-side failures (Timeout, Technical Error, Network Error).

In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49820 entries, 0 to 49819
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     49820 non-null  object        
 1   timestamp          49820 non-null  datetime64[ns]
 2   user_id            49820 non-null  object        
 3   device_id          49820 non-null  object        
 4   merchant_name      49820 non-null  object        
 5   merchant_category  49820 non-null  object        
 6   gateway_name       49820 non-null  object        
 7   amount             49820 non-null  float64       
 8   currency           49820 non-null  object        
 9   payment_method     49820 non-null  object        
 10  card_network       49820 non-null  object        
 11  issuer_bank        49820 non-null  object        
 12  device_type        49820 non-null  object        
 13  status             49820 non-null  object        
 14  failur

In [56]:
# Q13 (a): Check device_type categories and missing values
df['device_type'].value_counts(dropna=False)

device_type
Android    28950
iOS        12445
Web         8425
Name: count, dtype: int64

### Q13 (b): Success Rate by Device Type

**Business Question**
- Does the payment success rate differ between Android, iOS and Web checkouts?

**Why It Matters**
- Each platform has its own checkout flow. A weaker platform with high traffic would justify a fix to that flow, while similar rates mean the issue is not device-specific.


In [58]:
# Q13 (b): Success rate and volume share by device type
device_summary = (
    df.groupby('device_type')
      .agg(
          transactions=('is_success', 'count'),
          success_rate_pct=('is_success', 'mean')
      )
      .sort_values('success_rate_pct')
)

device_summary['success_rate_pct'] = (device_summary['success_rate_pct'] * 100).round(2)
device_summary['share_%'] = (device_summary['transactions'] / len(df) * 100).round(2)
device_summary

,transactions,success_rate_pct,share_%
device_type,,,
iOS,12445,77.47,24.98
Web,8425,78.01,16.91
Android,28950,78.34,58.11


**Insight**
- Success rates are very close across devices: iOS 77.47%, Web 78.01% and Android 78.34%, a gap of 0.87 percentage points.
- The iOS gap versus Android is borderline statistically, but practically very small.

**Business Impact**
- If iOS matched Android, only about 108 extra transactions (about 0.2 pp overall) would succeed, so a device-specific fix is not justified.

**Recommended Action**
- Do not prioritise device-specific fixes. Keep monitoring iOS in the dashboard in case the gap widens over time.

## Q14: Chargeback Analysis

**Business Question**
- How often do payments end in a chargeback, and does the chargeback pattern look valid?

**Why It Matters**
- A chargeback means the customer disputed the payment and the bank reversed it. The company loses the amount and may pay a penalty fee, and a high chargeback rate can lead card networks to restrict the account.
- Chargebacks can only occur on payments that actually succeeded, so the chargeback rate should be measured against successful transactions, not against all transactions.

**Metrics**
- Chargeback count (chargeback_flag = 1)
- Chargeback rate % = chargebacks ÷ successful transactions

In [61]:
# 14(a): check chargeback_flag values

df['chargeback_flag'].value_counts(dropna=False)

chargeback_flag
0    48667
1     1153
Name: count, dtype: int64

In [62]:
# 14(b): chargeback by transaction status

pd.crosstab(df['status'], df['chargeback_flag'])

chargeback_flag,0,1
status,,
Failed,6855,0
Pending,2546,0
Refunded,1488,38
Success,37778,1115


**Data note:** `status` has four values: Success (38,893), Failed (6,855), Pending (2,546) and Refunded (1,526). The success flag used so far counts only "Success", so Pending and Refunded are treated as not successful.

**Insight**
- All 1,153 chargebacks occur on charged payments (1,115 Success, 38 Refunded). None occur on Failed or Pending payments, which is consistent with how chargebacks work.
- The chargeback rate is about 2.85% of charged payments (1,153 of 40,419), well above the roughly 1% level at which card networks typically start monitoring merchants.

**Business Impact**
- At this rate, the company bears reversed amounts plus potential penalty fees, and risks account restrictions.
- The 38 Refunded payments with a chargeback may represent a double loss (refund plus chargeback) and should be reviewed.

**Recommended Action**
- Track the chargeback rate monthly against a 1% target.
- Review the 38 Refunded-and-disputed transactions to check for double payouts or a data inconsistency.

In [64]:
# Q14 (c): Chargeback rate on charged payments (Success + Refunded)
charged = df[df['status'].isin(['Success', 'Refunded'])]
chargeback_rate = charged['chargeback_flag'].mean() * 100

print(f"Charged payments: {len(charged):,}")
print(f"Chargebacks: {charged['chargeback_flag'].sum():,}")
print(f"Chargeback rate: {chargeback_rate:.2f}%")

Charged payments: 40,419
Chargebacks: 1,153
Chargeback rate: 2.85%


**Insight**
- Chargeback rate is 2.85% of charged payments (1,153 of 40,419), measured only on Success and Refunded payments because chargebacks cannot occur on Failed or Pending ones.
- This is well above the roughly 1% level at which card networks typically start monitoring merchants.

**Business Impact**
- Each chargeback reverses the payment and can add a penalty fee. At this rate the company also risks account restrictions from card networks.

**Recommended Action**
- Set a 1% target, track the rate monthly, and investigate which segments contribute most (next: value of chargebacks and risk score).

In [66]:
# Q14 (d): Value of chargebacks
chargeback_df = df[df['chargeback_flag'] == 1]
chargeback_amount = chargeback_df['amount'].sum()
charged_amount = charged['amount'].sum()

print(f"Chargeback amount: INR {chargeback_amount:,.2f}")
print(f"Share of charged value: {chargeback_amount / charged_amount * 100:.2f}%")

Chargeback amount: INR 464,626.94
Share of charged value: 2.83%


**Insight**
- Chargebacks total INR 464,627 over six months, which is 2.83% of charged value, in line with the 2.85% count-based rate. Chargebacks are therefore not concentrated in unusually high or low-value payments (average about INR 403 versus an overall average ticket of about INR 405).

**Business Impact**
- This is roughly INR 77,000 lost per month, before penalty fees, which makes chargebacks a larger and more concrete loss than any segment-level success-rate gap found so far.

**Recommended Action**
- Prioritise chargeback reduction over segment-level success-rate tuning.
- Identify where chargebacks concentrate (payment method, gateway, merchant category, risk score) to target prevention rules.

In [68]:
# Q14 (e): Chargeback rate by payment method (charged payments only)
cb_method = (
    charged.groupby('payment_method')
           .agg(
               charged_txns=('chargeback_flag', 'count'),
               chargebacks=('chargeback_flag', 'sum'),
               chargeback_rate_pct=('chargeback_flag', 'mean')
           )
           .sort_values('chargeback_rate_pct', ascending=False)
)

cb_method['chargeback_rate_pct'] = (cb_method['chargeback_rate_pct'] * 100).round(2)
cb_method

,charged_txns,chargebacks,chargeback_rate_pct
payment_method,,,
DEBIT CARD,6518,197,3.02
UPI,19464,564,2.90
NET BANKING,4041,114,2.82
CREDIT CARD,7996,218,2.73
WALLET,2400,60,2.50


**Insight**
- Chargeback rates are similar across payment methods, from 2.50% (Wallet) to 3.02% (Debit Card). The 0.52 pp gap is within random variation, so no payment method is a chargeback hotspot.
- The chargeback problem appears platform-wide rather than concentrated in one method.

**Business Impact**
- Method-specific chargeback controls would not reduce the overall rate meaningfully.

**Recommended Action**
- Do not target chargeback prevention by payment method.
- Test whether the risk score identifies chargebacks (next analysis), since an effective risk score would allow targeted prevention.

---

### Q14 Summary: Chargeback Analysis
- The chargeback rate is 2.85% of charged payments (1,153 of 40,419), worth about INR 4.65 lakh over six months and well above the roughly 1% monitoring level.
- Chargebacks occur only on Success and Refunded payments, as expected. 38 Refunded payments also have a chargeback and should be reviewed for double payout.
- The rate is uniform across payment methods, so the issue is platform-wide.

## Q15: Risk Score Effectiveness

**Business Question**
- Does the risk score separate payments that end in a chargeback from those that do not?

**Why It Matters**
- If chargeback payments carry clearly higher risk scores, the score can drive controls such as extra verification or manual review for high-risk payments, reducing chargeback losses.
- If the scores are similar, the risk score has no predictive value and the risk model needs review.

**Metrics**
- Average and median risk score by chargeback outcome
- Chargeback rate by risk band (low, medium, high)

**Scope**
- Only charged payments (Success and Refunded) are used, because chargebacks cannot occur on Failed or Pending payments.

In [71]:
# Q15(a): Understand risk_score scale
df['risk_score'].describe()

count    49820.000000
mean        35.113336
std         17.478077
min          0.000000
25%         22.920000
50%         34.915000
75%         46.810000
max        100.000000
Name: risk_score, dtype: float64

In [72]:
# Q15 (a2): Count scores at the scale limits
df['risk_score'].isin([0, 100]).sum()

1284

In [73]:
# Charged payments = Success + Refunded (chargebacks can occur only on these)
charged = df[df['status'].isin(['Success', 'Refunded'])]

# Q15 (b): Risk score by chargeback outcome (charged payments only)
charged.groupby('chargeback_flag')['risk_score'].agg(['count', 'mean', 'median'])

,count,mean,median
chargeback_flag,,,
0,39266,34.985981,34.915
1,1153,40.565199,37.180


**Insight**
- Payments with a chargeback have a higher average risk score (40.6) than payments without one (35.0), a gap of 5.6 points. With over 1,100 chargebacks this gap is not random, so the risk score carries real signal.
- The gap is modest compared with the overall spread of scores (about 17.5 points), so the two groups overlap heavily and the average alone does not show how useful the score is for decisions.

**Business Impact**
- The score can support targeted controls, but its benefit cannot be sized from averages. Chargeback rates by risk band are needed to see where a threshold would help.

**Recommended Action**
- Calculate the chargeback rate for low, medium and high risk bands (next step) and test whether extra verification on the high band is worthwhile.

In [75]:
# Q15 (c): Chargeback rate by risk band (charged payments only)
charged = charged.copy()
charged['risk_band'] = pd.cut(
    charged['risk_score'],
    bins=[-1, 30, 60, 100],
    labels=['Low', 'Medium', 'High']
)

band_summary = (
    charged.groupby('risk_band', observed=True)
           .agg(
               charged_txns=('chargeback_flag', 'count'),
               chargebacks=('chargeback_flag', 'sum'),
               chargeback_rate_pct=('chargeback_flag', 'mean')
           )
)
band_summary['chargeback_rate_pct'] = (band_summary['chargeback_rate_pct'] * 100).round(2)
band_summary

,charged_txns,chargebacks,chargeback_rate_pct
risk_band,,,
Low,15676,395,2.52
Medium,21446,535,2.49
High,3297,223,6.76


**Insight**
- Payments with a risk score above 60 (High band) have a chargeback rate of 6.76%, about 2.7 times the 2.5% rate of the Low and Medium bands. Low (2.52%) and Medium (2.49%) are almost identical, so the score only separates risk at the top end.
- The High band holds about 8% of charged payments but about 19% of all chargebacks (223 of 1,153).

**Business Impact**
- High-band chargebacks exceed the baseline rate by about 140, worth roughly INR 56,000 over six months, around 12% of total chargeback value. A High-band control would help but would not solve the overall chargeback problem.

**Recommended Action**
- Pilot extra verification (for example, step-up authentication) on payments with a risk score above 60 and measure the change in chargeback rate against any drop in successful payments.
- Review the risk model, since it does not distinguish Low from Medium risk.

---

### Q15 Summary: Risk Score
The risk score carries real signal, but only above a score of 60, where the chargeback rate is 6.76% versus about 2.5% elsewhere. Targeting that band is worthwhile, but it addresses only about 12% of chargeback value.

## Q16: MDR Fee Revenue

**Business Question**
- How much fee revenue does the platform earn, and how does it compare with chargeback losses?

**Why It Matters**
- MDR (Merchant Discount Rate) fees are the platform's main revenue. Comparing them with chargeback losses shows whether the business is healthy, and which gateways or payment methods earn the most.

**Metrics**
- Total MDR fee revenue
- Effective MDR rate = total fee ÷ total amount of fee-bearing payments
- Chargeback loss as a share of MDR revenue

In [78]:
# Q16(a): MDR fee by transaction status
df.groupby('status')['mdr_fee'].agg(['count', 'sum', 'mean'])

,count,sum,mean
status,,,
Failed,6855,43173.47,6.298099
Pending,2546,15497.96,6.087180
Refunded,1526,9980.56,6.540341
Success,38893,241802.32,6.217117


**Data note:** `mdr_fee` is populated for every payment, including Failed (INR 43,173) and Pending (INR 15,498) payments that earned no fee. It therefore represents an expected fee, not realised revenue. MDR revenue is measured on Success payments only; Refunded payments are excluded because the data does not show whether their fee was returned.

In [80]:
# Q16(b): MDR revenue on successful payments and effective MDR rate
paid = df[df['status'] == 'Success']

total_fee = paid['mdr_fee'].sum()
fee_base = paid['amount'].sum()
effective_rate = total_fee / fee_base * 100

print(f"Succcessful payments: {len(paid):,}")
print(f"payment value : INR {fee_base:,.2f}")
print(f"Total MDR revenue : INR {total_fee:,.2f}")
print(f"Effective MDR rate : {effective_rate:.2f}%")

Succcessful payments: 38,893
payment value : INR 15,727,325.38
Total MDR revenue : INR 241,802.32
Effective MDR rate : 1.54%


**Insight**
- Successful payments worth INR 15.73 million earned MDR revenue of INR 241,802 over six months, an effective MDR rate of 1.54% (about INR 1.54 per INR 100 processed).
- The average fee is about INR 6.2 per successful payment.

**Business Impact**
- Revenue is roughly INR 40,000 per month. A 0.1 percentage point change in the effective rate would change revenue by about INR 15,700 over six months.

**Recommended Action**
- Track the effective MDR rate by gateway and payment method to find where the platform earns the most per rupee.
- Report fee revenue on Success payments only, since the `mdr_fee` column also holds fees for Failed and Pending payments.

In [82]:
# Q16 (c): Chargeback loss versus MDR revenue
net_after_chargebacks = total_fee - chargeback_amount

print(f"MDR revenue           : INR {total_fee:,.2f}")
print(f"Chargeback loss       : INR {chargeback_amount:,.2f}")
print(f"Net after chargebacks : INR {net_after_chargebacks:,.2f}")
print(f"Loss as % of revenue  : {chargeback_amount / total_fee * 100:.1f}%")

MDR revenue           : INR 241,802.32
Chargeback loss       : INR 464,626.94
Net after chargebacks : INR -222,824.62
Loss as % of revenue  : 192.2%


**Insight**
- Chargeback losses (INR 464,627) are about 1.9 times the MDR fee revenue (INR 241,802) earned over six months. Chargebacks alone more than wipe out the fee income, leaving a net shortfall of about INR 223,000.
- For every INR 100 of successful payments the platform earns about INR 1.54 in fees but loses about INR 2.95 to chargebacks.

**Business Impact**
- On these figures the payment business is loss-making before any operating cost, bank or network share, or chargeback penalty fees, all of which would widen the gap.

**Recommended Action**
- Treat chargeback reduction as the top priority: pilot extra verification on payments with a risk score above 60, where the chargeback rate is 6.76%, and review the 38 Refunded payments that also have a chargeback.
- Review pricing and merchant contracts, for example a chargeback fee passed on to merchants or a higher rate for high-risk payments.

**Note:** MDR is gross fee revenue, not profit, and chargeback loss is counted at the full payment amount without penalty fees or recovered disputes. The dataset is synthetic, so the size of the gap is illustrative.

In [84]:
# Q17 (a): Gateway scorecard, part 1

# Step 1: Failed flag
df['is_failed'] = (df['status'] == 'Failed').astype(int)

# Step 2 + 3: group and aggregate
scorecard = (
    df.groupby('gateway_name')
      .agg(
          transactions=('is_success', 'count'),
          success_rate_pct=('is_success', 'mean'),
          failed_rate_pct=('is_failed', 'mean'),
          avg_response_ms=('response_time_ms', 'mean')
      )
)

# Step 4: percentage + round
scorecard['success_rate_pct'] = (scorecard['success_rate_pct'] * 100).round(2)
scorecard['failed_rate_pct'] = (scorecard['failed_rate_pct'] * 100).round(2)
scorecard['avg_response_ms'] = scorecard['avg_response_ms'].round(0)

scorecard

,transactions,success_rate_pct,failed_rate_pct,avg_response_ms
gateway_name,,,,
Cashfree,13274,78.24,13.27,717.0
PayU,11353,78.31,13.74,717.0
Razorpay,19712,78.06,13.96,716.0
Stripe,4983,77.00,14.39,725.0
Unknown,498,78.92,13.05,703.0


### Q17 (a): Gateway Scorecard - Volume, Success, Failure and Speed

**Insight**
- All gateways perform almost identically: success rate 77.00-78.92%, failed rate 13.05-14.39%, response time 703-725 ms.
- Stripe is the lowest (77.00% success), but the gap vs Razorpay (78.06%) is about 1 point on 4,983 transactions, which is within normal random variation.

**Business Impact**
- Moving traffic between gateways is unlikely to change the success rate. If Stripe matched the average, it would add only about 50 successful payments in six months.
- 498 transactions (1%) have an Unknown gateway, so they cannot be attributed to any gateway.

**Recommended Action**
- Do not re-route traffic based on success rate alone. Compare gateways on chargeback rate and fee revenue next (Q17 b).
- Fix the capture of gateway_name so the Unknown rows can be traced.

In [86]:
# Q17 (b): Gateway scorecard, part 2: chargeback and MDR revenue

# Step 1: chargeback by gateway (denominator = charged payments)
cb_gateway = (
    charged.groupby('gateway_name')
           .agg(
               charged_txns=('chargeback_flag', 'count'),
               chargebacks=('chargeback_flag', 'sum'),
               chargeback_rate_pct=('chargeback_flag', 'mean')
           )
)
cb_gateway['chargeback_rate_pct'] = (cb_gateway['chargeback_rate_pct'] * 100).round(2)

# Step 2: MDR revenue by gateway (Success payments only)
fee_gateway = (
    paid.groupby('gateway_name')
        .agg(
            mdr_revenue=('mdr_fee', 'sum'),
            success_amount=('amount', 'sum')
        )
)

# Step 3: effective MDR rate
fee_gateway['effective_rate_pct'] = (
    fee_gateway['mdr_revenue'] / fee_gateway['success_amount'] * 100
).round(2)
fee_gateway['mdr_revenue'] = fee_gateway['mdr_revenue'].round(2)

display(cb_gateway)
fee_gateway

,charged_txns,chargebacks,chargeback_rate_pct
gateway_name,,,
Cashfree,10812,311,2.88
PayU,9225,257,2.79
Razorpay,15970,455,2.85
Stripe,4003,119,2.97
Unknown,409,11,2.69


,mdr_revenue,success_amount,effective_rate_pct
gateway_name,,,
Cashfree,64450.38,4180344.20,1.54
PayU,55615.47,3620443.44,1.54
Razorpay,95701.33,6236708.19,1.53
Stripe,23669.71,1539060.86,1.54
Unknown,2365.43,150768.69,1.57


### Q17 (b): Chargeback and MDR Revenue by Gateway

**Insight**
- Chargeback rate is almost identical across gateways (2.69-2.97%, overall 2.85%). Stripe is the highest (2.97%), but the gap is about 0.1 point on 4,003 payments, which is within random variation.
- Effective MDR rate is flat at about 1.53-1.54% for every gateway, so no gateway earns more fee per rupee processed.
- MDR revenue simply follows volume: Razorpay ~40%, Cashfree ~27%, PayU ~23%, Stripe ~10%.

**Business Impact**
- Changing the gateway mix will not reduce chargebacks or increase fee income.
- Razorpay alone brings about ₹95.7K of ₹241.8K fee revenue, so any outage or contract change there affects ~40% of revenue.

**Recommended Action**
- Do not use gateway choice to fix chargebacks. Focus on risk-score controls instead (Q15 finding).
- Keep a backup routing plan for Razorpay to reduce dependency.

In [88]:
# Q17 (c): Final gateway scorecard

# Step 1: chargeback amount by gateway
cb_amount = (
    chargeback_df.groupby('gateway_name')['amount']
                 .sum()
                 .round(2)
                 .rename('chargeback_amount')
)

# Step 2: join everything on gateway_name
scorecard_full = (
    scorecard
    .join(cb_gateway[['chargeback_rate_pct']])
    .join(fee_gateway[['mdr_revenue', 'effective_rate_pct']])
    .join(cb_amount)
)

# Step 3: chargeback loss vs fee revenue
scorecard_full['loss_to_mdr_pct'] = (
    scorecard_full['chargeback_amount'] / scorecard_full['mdr_revenue'] * 100
).round(1)

scorecard_full

,transactions,success_rate_pct,failed_rate_pct,avg_response_ms,chargeback_rate_pct,mdr_revenue,effective_rate_pct,chargeback_amount,loss_to_mdr_pct
gateway_name,,,,,,,,,
Cashfree,13274,78.24,13.27,717.0,2.88,64450.38,1.54,124429.04,193.1
PayU,11353,78.31,13.74,717.0,2.79,55615.47,1.54,99096.74,178.2
Razorpay,19712,78.06,13.96,716.0,2.85,95701.33,1.53,191744.49,200.4
Stripe,4983,77.00,14.39,725.0,2.97,23669.71,1.54,46402.10,196.0
Unknown,498,78.92,13.05,703.0,2.69,2365.43,1.57,2954.57,124.9


### Q17 (c): Final Gateway Scorecard

**Insight**
- All gateways look the same on every metric: success rate 77.00-78.92%, chargeback rate 2.69-2.97%, effective MDR about 1.54%, response time about 716 ms. No gateway is clearly better or worse.
- Every gateway loses more to chargebacks than it earns in MDR: chargeback loss is 178-200% of MDR revenue for the four named gateways.

**Business Impact**
- The profit leak is a business-wide chargeback problem, not a gateway problem. Re-routing traffic will not fix it.
- Razorpay has the largest absolute loss (about Rs 191.7K, ~41% of chargeback value) simply because it carries the most volume.

**Recommended Action**
- Focus on chargeback prevention, starting with step-up verification for payments with risk score above 60 (Q15 finding).
- Keep the current gateway mix, and review fee and penalty terms and Razorpay dependency separately.

In [141]:
scorecard_full.to_csv('gateway_scorecard.csv')

## Executive Summary: Payment Gateway Performance & Risk Analytics

**Headline**

Chargebacks, not gateway or payment-method failures, are the biggest profit leak: chargeback losses (Rs 464.6K) are about 1.9x the MDR fee revenue (Rs 241.8K) in Jan-Jun 2026, leaving a net loss of Rs 222.8K.

**Evidence**
- Success rate is about 78% and stays flat across gateway (77.0-78.9%), payment method, issuer bank and device. No segment explains the failures.
- Chargeback rate is 2.85% (1,153 of 40,419 charged payments), while the effective MDR rate is only 1.54% of Rs 15.73M processed.
- Payments with risk score above 60 have a 6.76% chargeback rate versus about 2.5% for low and medium risk. They are about 8% of payments but about 19% of chargebacks.
- All gateways look alike on success rate, chargeback rate, fee rate and speed, so no gateway is clearly better or worse.

**Business Impact**
- Every Rs 100 of fee income is followed by about Rs 192 of chargeback loss.
- The excess chargebacks in the high-risk band are about 140 (about Rs 56K, roughly 12% of chargeback value). Fixing this band helps, but will not close the gap alone.

**Recommendation**
1. Pilot step-up verification (e.g., OTP/3DS) for risk score above 60. KPI: high-band chargeback rate; guardrail: success rate. Priority: High.
2. Review the 38 payments that were both refunded and charged back (possible double loss). Priority: High, low effort.
3. Review pricing and chargeback terms: chargeback losses (Rs 464.6K) are about 1.9x the fee revenue (Rs 241.8K). Priority: Medium.
4. Do not re-route traffic between gateways. It will not change the outcome. Keep a backup plan for Razorpay, which carries about 40% of fee revenue.

**Confidence and Limitations**
- Direction is reliable, exact amounts are not: the data is synthetic, covers 6 months, MDR is gross, and chargeback penalty fees are not included.
- 1% of transactions have an Unknown gateway.
- Gateway-level differences in chargeback loss rest on small samples (119-455 chargebacks per gateway) and should not be over-read.